# AVSR · GRID · пайплайн для курсовой

Запускать ячейки **по порядку сверху вниз**. Раздел A — один раз за сессию. Разделы B/C/D — собственно эксперименты.

| Раздел | Что делает | Время |
|---|---|---|
| **A. SETUP** | клон репозитория + зависимости | ~1 мин |
| **A. DATA** | скачать GRID (s1,s2) + губы | ~13 мин |
| **B. 4.4/4.6** | zero-shot Whisper: WER/CER по SNR + RTF | ~5 мин |
| **C. LAUNCH** | обучить 3 модели (audio/AV) в фоне | в фоне |
| **D. MONITOR** | смотреть прогресс и результаты | по запросу |

Обучение в разделе C идёт **в фоне** и переживает обрывы соединения. Если рантайм отвалился — заново выполни A.SETUP (данные в `/content/grid` тоже придётся пересоздать, если VM сменилась) и снова C.LAUNCH: обучение продолжится с последнего чекпоинта.

## A. SETUP — клон репозитория и зависимости

In [ ]:
!rm -rf /content/coursera && git clone -q https://github.com/DanKolganov/coursera.git /content/coursera
!pip install -q mediapipe jiwer einops omegaconf transformers
print('SETUP OK')

## A. DATA — скачать GRID (s1, s2) и извлечь губы
Пропускает скачивание, если данные уже на месте.

In [ ]:
import os
if os.path.exists('/content/grid/manifests/train.jsonl'):
    print('Данные уже есть — пропускаю.')
else:
    !cd /content/coursera && PYTHONPATH=. python scripts/prepare_grid.py \
        --speakers s1,s2 --base /content/grid --val-speakers s2 \
        --detect-every 5 --delegate gpu

## B. Таблицы 4.4 + 4.6 — zero-shot Whisper (без обучения)
Прогоняет предобученный `whisper-small` по GRID с разным уровнем шума (SNR) и меряет скорость (RTF). Результат — две таблицы для главы 4.

In [ ]:
import warnings, re, json, time, math
warnings.filterwarnings('ignore')
import numpy as np, torch, soundfile as sf
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers.utils import logging as _l; _l.set_verbosity_error()

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
proc = WhisperProcessor.from_pretrained('openai/whisper-small')
model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small').to(dev).eval()

N = 150
recs = [json.loads(l) for l in open('/content/grid/manifests/val.jsonl')][:N]
D2W = {'0':'zero','1':'one','2':'two','3':'three','4':'four','5':'five','6':'six','7':'seven','8':'eight','9':'nine'}
def norm(s):
    s = re.sub(r'[^a-z0-9 ]',' ', s.lower())
    return ' '.join(D2W.get(t,t) for t in s.split())
def ed(r,h):
    r,h=r.split(),h.split(); d=[[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0]=i
    for j in range(len(h)+1): d[0][j]=j
    for i in range(1,len(r)+1):
        for j in range(1,len(h)+1):
            d[i][j]=min(d[i-1][j]+1,d[i][j-1]+1,d[i-1][j-1]+(r[i-1]!=h[j-1]))
    return d[len(r)][len(h)],len(r)
def ced(r,h):
    r,h=list(r),list(h); d=[[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0]=i
    for j in range(len(h)+1): d[0][j]=j
    for i in range(1,len(r)+1):
        for j in range(1,len(h)+1):
            d[i][j]=min(d[i-1][j]+1,d[i][j-1]+1,d[i-1][j-1]+(r[i-1]!=h[j-1]))
    return d[len(r)][len(h)],len(r)
def noise(w,snr):
    if snr is None: return w
    pn=(np.mean(w**2)+1e-12)/(10**(snr/10)); return w+np.random.randn(len(w)).astype(np.float32)*math.sqrt(pn)
wavs=[sf.read(r['audio'])[0].astype(np.float32) for r in recs]
wavs=[w.mean(1) if w.ndim>1 else w for w in wavs]

print('SNR     WER%    CER%'); rt_p=rt_a=0.0
for snr in [None,20,15,10,5,0]:
    we=wt=ce=ct=0
    for r,w in zip(recs,wavs):
        f=proc(noise(w,snr),sampling_rate=16000,return_tensors='pt').input_features.to(dev)
        t0=time.time()
        with torch.no_grad(): ids=model.generate(f,language='en',task='transcribe',max_new_tokens=40,num_beams=1)
        if snr is None: rt_p+=time.time()-t0; rt_a+=len(w)/16000
        h=norm(proc.batch_decode(ids,skip_special_tokens=True)[0]); g=norm(r['text'])
        e,t=ed(g,h); we+=e; wt+=t
        e,t=ced(g,h); ce+=e; ct+=t
    tag='clean' if snr is None else f'{snr}dB'
    print(f'{tag:6s}  {100*we/max(wt,1):6.1f}  {100*ce/max(ct,1):6.1f}')
print(f'\nRTF (whisper-small, clean, bs=1) = {rt_p/max(rt_a,1e-9):.4f}  | {1000*rt_p/len(recs):.0f} ms/clip')

## C. Таблица 4.5 — обучить 3 модели (фоновый запуск)
Обучает `audio_only`, `av_concat`, `av_cross` (замороженный Whisper + CTC) и считает WER/CER по SNR. Запуск **в фоне**: переживает обрывы. Результаты пишутся в `results/*.json`.

In [ ]:
!cd /content/coursera && PYTHONPATH=. nohup python -u scripts/run_grid_experiments.py \
    --base /content/grid > /content/exp.log 2>&1 & echo 'LAUNCHED pid' $!

## D. MONITOR — прогресс и результаты
Выполняй эту ячейку повторно, чтобы следить за ходом. `summary.json` появляется по мере готовности моделей.

In [ ]:
!tail -n 20 /content/exp.log
print('\n===== РЕЗУЛЬТАТЫ (results/) =====')
!for f in /content/coursera/results/*.json; do echo "--- $f ---"; cat "$f"; echo; done 2>/dev/null || echo 'результатов пока нет'